In [ ]:
import os
import random
import numpy as np
from tqdm.notebook import tqdm
import imageio
from IPython.display import Image, display
from typing import Dict
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image as pil_Image
import torch
import io
import PIL

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def enable_determinism():
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)


def fix_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.mps.manual_seed(seed)


# https://pytorch.org/docs/stable/notes/randomness.html#dataloader
def seed_worker(_):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed = 28
enable_determinism()
fix_seeds(seed)
generator = torch.Generator()
generator.manual_seed(seed)

# > Генерация. Диффузионные модели.

Мы изучили теорию и на практике реализовали процесс генерации изображений с помощью диффузионных моделей. Теперь давайте подробнее познакомимся с фреймворком diffusers, предназначенным для работы с диффузионными моделями.

План:
- Настройка основных параметров генерации изображений;
- Создание промптов с использованием векторного представления текста;
- Разработка кастомных callback-классов.
- Инерполяция латентных векторов.

# > Настройка основных параметров генерации изображений

Для начала загрузим модель и проверим что все работает.

In [ ]:
pipeline = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")
pipeline.to(device);


In [ ]:
prompt = "person in front of a beautiful waterfall"
pipeline(prompt).images[0]

Давайте подробнее рассмотрим параметры, которые можно задать при генерации изображения:

- prompt: текстовое описание, на основе которого будет генерироваться изображение.
- negative_prompt: текстовое описание, которое будет исключено из процесса генерации, помогая улучшить результаты.
- height: высота генерируемого изображения в пикселях.
- width: ширина генерируемого изображения в пикселях.
- num_inference_steps: количество шагов моделирования для получения итогового изображения; большее количество шагов может привести к более точным результатам, но увеличит время генерации.
- num_images_per_prompt: количество изображений, генерируемых на основе одного промпта.
- guidance_scale: коэффициент, определяющий, насколько сильно промпт будет влиять на процесс генерации; более высокие значения усиливают соответствие изображения промпту.
- generator: источник случайных чисел, используемый для воспроизводимости результатов генерации.

Теперь можно поэкспериментировать с параметрами и посмотреть, какие результаты получаются. C точки зрения обучения это не несет большого смысла, но это довольно весело.

Главное — вовремя остановиться, чтобы не исчерпать все лимиты в Google Colab :)

In [ ]:
prompt = "person in front of a beautiful waterfall"
negative_prompt = "forest, tree, green"
height = 256
width = 512
num_inference_steps = 100
num_images_per_prompt = 4
guidance_scale = 7.5


images = pipeline(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=height,
    width=width,
    num_inference_steps=num_inference_steps,
    num_images_per_prompt=num_images_per_prompt,
    guidance_scale=guidance_scale,
    generator=generator,
    ).images

for image in images:
    display(image)


# > Создание промптов с использованием векторного представления текста

 В качестве промпта, помимо текста, можно передавать векторы, полученные от текстового энкодера модели. Это позволяет делать довольно интересные вещи, например, "играть со смыслами" за счет математических операций с векторами.

 Для начала возьмем из нашей модели tokenize и text_encoder.

In [ ]:
tokenizer = pipeline.tokenizer
text_encoder = pipeline.text_encoder

Теперь нам нужно разбить текст на токены и получить векторное представление для текстового запроса.

**Задание 1**. Реализуйте функцию get_prompt_embedding, которая будет выдавать эмбеддинг для текстового запроса.

In [ ]:
# QWEN
def get_prompt_embedding(prompt_text, tokenizer, text_encoder, device=device):
    # Получите токены, из которых состоит текстовый запрос
    # padding="max_length" и max_length=tokenizer.model_max_length (77 для SD 1.5) 
    # гарантируют фиксированный размер выхода, truncation=True обрезает слишком длинные запросы
    tokenized_inputs = tokenizer(
        prompt_text, 
        padding="max_length", 
        max_length=tokenizer.model_max_length, 
        truncation=True
    )
    
    # Преобразование индексов токенов в нужный формат
    input_ids_tensor = torch.Tensor([tokenized_inputs.input_ids]).to(device).long()
    
    # Получите эмбеддинг на основе индексов токенов
    text_embeddings = text_encoder(input_ids_tensor)
    
    # Извлечение последнего скрытого состояния из эмбеддингов
    prompt_embedding = text_embeddings.last_hidden_state
    
    return prompt_embedding

Давайте теперь посмотрим на классический пример, как из короля сделать королеву.

In [ ]:
king_prompt = 'king'
man_prompt = 'man'
woman_prompt = 'woman'

In [ ]:
king_embedding = get_prompt_embedding(king_prompt, tokenizer, text_encoder, device)
man_embedding = get_prompt_embedding(man_prompt,  tokenizer, text_encoder, device)
woman_embedding = get_prompt_embedding(woman_prompt,  tokenizer, text_encoder, device)

In [ ]:
king_embedding

**Задание 2**. Напишите функцию get_queen_embedding, которая получит эмбединг описывающий королеву, на основе математических операций сложения и вычитания векторов для "мужчины", " короля" и "женщины".

In [ ]:
def get_queen_embedding(king_embedding, man_embedding, woman_embedding):
  qe = king_embedding + woman_embedding - man_embedding
  return qe

In [ ]:
queen_embedding = get_queen_embedding(king_embedding, man_embedding, woman_embedding)

In [ ]:
queen_embedding

Ради интереса теперь можно посмотреть что у нас получилось.
> Тут можно примять все настройки из прошлого раздела, но для простоты я буду использовать самый простой запрос

In [ ]:
# король
pipeline(prompt_embeds=king_embedding).images[0]

In [ ]:
# мужчина
pipeline(prompt_embeds=man_embedding).images[0]

In [ ]:
# Девушка
pipeline(prompt_embeds=woman_embedding).images[0]


In [ ]:
# Королева
pipeline(prompt_embeds=queen_embedding).images[0]


# > Разработка кастомных callback-классов

Библиотека diffusers позваляет добавлять к модели кастомные callback-функции для изменения поведения модели.

Давайте посмотрим как это работает и напишим простой callback, который будет просто сохранять латентные вектра на каждом шаге диффузии. Это может быть полезно для визуализации процесса диффузии и отладки модели.

Для этой цели мы можем использовать параметр callback_on_step_end, куда передим свой callback. Переданный нами callback будет выполнятся в конце каждого шага диффузии.


**Задание 3**. Реализуйте класс SaveLatentVectorsCallback, который будет на каждом шаге сохранять латентные вектора. Данный класс должен иметь метод который позволяют получить накопленные вектора и метод который позволяет удалить их.

In [ ]:
class SaveLatentVectorsCallback:
    def __init__(self):
        # инициализирует список для хранения латентных векторов
        self.latents_collection = []

    def __call__(self, pipeline, step: int, timestep: int, callback_kwargs: dict):
        latents = callback_kwargs["latents"]
        # сохраняет латентный вектор при каждом вызове функции
        # Используем .clone(), чтобы отвязать сохраненный тензор от графа вычислений,
        # и .cpu(), чтобы предотвратить переполнение видеопамяти (VRAM) при сохранении многих шагов.
        self.latents_collection.append(latents.clone().cpu())
        
        return {"latents": latents}

    def get_latent_vectors(self):
        # возвращает список латентных векторов, накопленных во время инференса
        return self.latents_collection

    def clean_latent_vectors(self):
        # очищает список латентных векторов
        self.latents_collection.clear()

Инициализируем наш callback.

In [ ]:
save_latent_vectors_callback = SaveLatentVectorsCallback()

In [ ]:
save_latent_vectors_callback.clean_latent_vectors()

# Определяем список тензоров, которые будут переданы в callback
callback_on_step_end_tensor_inputs = ["latents"]

pipeline(
    # "City lights of evening New York",
    "Red Maine Coon walking in Woods",
    callback_on_step_end=save_latent_vectors_callback,
    callback_on_step_end_tensor_inputs=callback_on_step_end_tensor_inputs,
    num_inference_steps=50
).images[0]

Мы сохранили латентные вектора, теперь нам нужно научиться декодировать их, что бы получить изображение.

Для этого нам понадобится энкодер.

In [ ]:
vae = pipeline.vae


**Задание 4**. Реализуйте функцию decode_image, которая на вход принимает латентный вектор и модель энкодера, а на выход выдает декодированное изображение.

In [ ]:
def decode_image(latents, vae):
    with torch.no_grad():
        # Переносим латенты на то же устройство, что и VAE
        latents = latents.to(vae.device)
        
        # Масштабируем латенты обратно
        latents = 1 / vae.config.scaling_factor * latents
        
        # Декодируем изображение с помощью модели VAE
        image = vae.decode(latents).sample
        
        # Денормализуем значения из диапазона [-1, 1] в [0, 1]
        image = (image / 2 + 0.5).clamp(0, 1)
        
        # Переносим на CPU, меняем порядок осей с (B, C, H, W) на (B, H, W, C) 
        # и преобразуем в numpy массив
        image = image.cpu().permute(0, 2, 3, 1).float().numpy()
        
        # Извлекаем первое изображение из батча (удаляем размерность батча)
        image = image[0]
        
    return image

In [ ]:
# Решение ,которое прошло чекер:
def decode_image(latents, vae):   
    with torch.no_grad():    
        latents = latents.to(vae.device)
        latents = 1 / vae.config.scaling_factor * latents    
        image = vae.decode(latents).sample
        # image = image[0]     
        image = (image / 2 + 0.5).clamp(0, 1)      
        image = image.cpu().permute(0, 2, 3, 1).float().numpy()
    return image

# Решение автора:
def decode_image(latents, vae):
  with torch.no_grad():
    latents = 1 / vae.config.scaling_factor * latents
    image = vae.decode(latents)
    image = image[0]
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()
  return image


Реализуем вспомогательные функции, что бы декодировать все сохраненные вектора и создать gif с визуализацией процеса диффузии.

In [ ]:
def decode_latents_collections(latents_collection, vae):
  images = []
  for latents in tqdm(latents_collection):
      image = decode_image(latents, vae)
      # image = np.squeeze(image, axis=0)
      images.append((image * 255).astype(np.uint8))
  return images

def create_gif(images, filename='diffusion_progress_NY_MC_HM.gif'):
  imageio.mimsave(filename, images, fps=10, loop=1)
  display(Image(filename=filename))

Давайте посмотрим что получилось.

In [ ]:
latents_collection = save_latent_vectors_callback.latents_collection
images = decode_latents_collections(save_latent_vectors_callback.latents_collection, pipeline.vae)
create_gif(images)


Попробуем провести эксперимент с процессом диффузии, используя callback для изменения латентного вектора на каждом шаге.

Например, чтобы создать симметричные изображения, можно отразить наше латентное представление по горизонтали: отразить верхнюю половину вектора и заменить ею нижнюю часть.

**Задание 5**. Реализуйте класс SymmetryCallback. На каждом шаге диффузии наш callback должен брать латентное представление и отражать его по горизонтали.

In [ ]:
x = torch.tensor([1, 2, 3])
y = torch.zeros_like(x)
y

In [ ]:
class SymmetryCallback(SaveLatentVectorsCallback):
    def __call__(self, pipeline, step: int, timestep: int, callback_kwargs):
        latents = callback_kwargs["latents"]
        batch_size, num_channels, height, width = latents.size()
        
        # 1. Клонируем тензор, чтобы не изменять исходные данные inplace (хорошая практика)
        modified_latents = latents.clone()
        
        # 2. Извлекаем верхнюю половину латентного представления
        # Форма: (batch_size, num_channels, height // 2, width)
        upper_half = modified_latents[:, :, :height // 2, :]
        
        # 3. Отражаем верхнюю половину по оси высоты (dim=2). 
        # Это создаст идеальное зеркальное отражение для нижней части.
        flipped_upper_half = torch.flip(upper_half, dims=[2])
        
        # 4. Заменяем нижнюю половину на отраженную верхнюю часть
        # Срез [:, :, height // 2 :, :] имеет точно такую же форму (..., height // 2, ...), 
        # как и flipped_upper_half, поэтому ошибка размерностей невозможна.
        modified_latents[:, :, height // 2 :, :] = flipped_upper_half

        # 5. Сохраняем модифицированный латентный вектор
        self.latents_collection.append(modified_latents.clone().cpu())
        
        return {"latents": modified_latents}

Посмотрим что из этого получилось

In [ ]:
symmetry_callback = SymmetryCallback()

pipeline(
    "Red Maine Coon walking in Woods",
    callback_on_step_end=symmetry_callback,
    callback_on_step_end_tensor_inputs=callback_on_step_end_tensor_inputs,
    num_inference_steps=50
)

latents_collection = symmetry_callback.latents_collection
images = decode_latents_collections(symmetry_callback.latents_collection, pipeline.vae)
create_gif(images, filename='symmetry-MC.gif')

In [ ]:
# Решение автора
class SymmetryCallback(SaveLatentVectorsCallback):
    def __call__(self, pipeline, step: int, timestep: int, callback_kwargs):
        latents = callback_kwargs["latents"]
        batch_size, num_channels, height, width = latents.size()
        # Создайте маску из 0, совпадающую по размеру с латентным представлением изображения.
        mask = torch.zeros((height, width), device=latents.device)

        mask[:height // 2] = 1
        expanded_mask = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, num_channels, -1, -1)

        # Извлекаем верхнюю половину и отражаем по горизонтальной оси
        upper_half = latents * expanded_mask
        flipped_upper_half = torch.flip(upper_half, dims=[2])  # Отражение по вертикальной оси
        # Создаём финальный результат, комбинируя верхнюю часть с отражённой верхней частью внизу
        modified_latents = latents * expanded_mask + flipped_upper_half * (1 - expanded_mask)

        # Сохраняем промежуточный латентный вектор, так же как в SaveLatentVectorsCallback
        self.latents_collection.append(modified_latents)

        return {"latents": modified_latents}




# > Инерполяция латентных векторов

Теперь, когда мы умеем сохранять латентные вектора, можно сгенерировать плавный переход от одного изображение к другому за счет интерполяции латентных векторов от разных изображений.

Для начала нам нужно создать несколько изображений и сохранить их латентные представления.


In [ ]:
save_latent_vectors_callback.clean_latent_vectors()
num_images = 10
latents = []
for i in range(num_images):
    pipeline(
      # "photorealistic image of galaxy and space",
      "Red Maine Coon walking in Woods",
      callback_on_step_end=save_latent_vectors_callback,
      callback_on_step_end_tensor_inputs=callback_on_step_end_tensor_inputs,
      num_inference_steps=50
      )
    latent = save_latent_vectors_callback.latents_collection[-1]
    latents.append(latent)

**Задание 6**. Реализуйте функцию interpolate_vectors, которая выполняет линейную интерполяцию между двумя латентными векторами latent_1 и latent_2 c коэффициентом alpha.

Для каждой пары  векторов выполните линейную интерполяцию на заданное количество шагов.

In [ ]:
def interpolate_vectors(latent_1, latent_2, alpha):
    latent_interpolated = (1 - alpha) * latent_1 + alpha * latent_2
    return latent_interpolated

def interpolate_latents(latent_vectors, steps, vae):
    images = []

    # Проверяем, что минимум два латентных вектора передано
    if len(latent_vectors) < 2:
        raise ValueError("Необходимо передать минимум два латентных вектора для интерполяции.")

    # Проходим по всем парам латентных векторов
    for i in range(len(latent_vectors) - 1):
        image_1_latent = latent_vectors[i]
        image_2_latent = latent_vectors[i + 1]

        for alpha in tqdm(np.linspace(0, 1, steps), desc=f'Interpolating between vector {i} and {i + 1}'):
            latent_interpolated = interpolate_vectors(image_1_latent, image_2_latent, alpha)
            image = decode_image(latent_interpolated, vae)
            # image = np.squeeze(image, axis=0)
            images.append((image * 255).astype(np.uint8))
    return images

steps = 25
images = interpolate_latents(latents, steps, vae)
create_gif(images,filename="interpolate-MC.gif")